# 🇳🇵 Chatterbox Nepali TTS Demo

Fine-tuned Nepali text-to-speech with voice cloning.

**Model**: [Imbatmann/chatterbox-nepali-tts](https://huggingface.co/Imbatmann/chatterbox-nepali-tts)

⚠️ Change runtime type to **T4 GPU** for best results: `Runtime → Change runtime type → T4 GPU`

In [ ]:
# 📦 Install dependencies
!pip install -q torch torchaudio --index-url https://download.pytorch.org/whl/cu121
!git clone https://github.com/45Harry/chatterbox-nepali.git /tmp/cb
!cp /tmp/cb/pyproject_colab.toml /tmp/cb/pyproject.toml
!pip install -q -e /tmp/cb safetensors

In [ ]:
# 🤖 Load Model
import torch, torchaudio as ta
from chatterbox.mtl_tts import ChatterboxMultilingualTTS
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file
from IPython.display import Audio, display

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Load base model
model = ChatterboxMultilingualTTS.from_pretrained(device)

# Load Nepali fine-tuned weights
ckpt = hf_hub_download("Imbatmann/chatterbox-nepali-tts", "t3_mtl_nepali_final.safetensors")
sd = load_file(ckpt)
cleaned = {k.replace("patched_model.", "").replace("model.", ""): v for k, v in sd.items()}
model.t3.load_state_dict(cleaned, strict=False)
model.t3.to(device).eval()
print("✅ Model loaded!")

In [ ]:
# 🎤 Download reference audio
!wget -q https://huggingface.co/Imbatmann/chatterbox-nepali-tts/resolve/main/ref.wav -O ref.wav
print("✅ Reference audio downloaded")
display(Audio("ref.wav"))

In [ ]:
# 🗣️ Generate Nepali Speech
text = "नमस्ते, म नेपाली एआई हुँ। मलाई तपाईंसँग कुरा गर्न पाउँदा खुसी लागेको छ।"
print(f"Generating: {text}")

wav = model.generate(
    text=text,
    language_id="ne",
    audio_prompt_path="ref.wav",
    exaggeration=0.5,
    temperature=0.8,
)

ta.save("output.wav", wav, model.sr)
print(f"✅ Generated: {wav.shape[1]/model.sr:.1f}s of audio")
display(Audio("output.wav"))

In [ ]:
# 🔁 Try different texts
texts = [
    "नेपाल हिमाल, पहाड र तराईले भरिएको सुन्दर देश हो।",
    "काठमाडौं उपत्यकाको ऐतिहासिक र सांस्कृतिक महत्त्व धेरै ठूलो छ।",
    "नेपाली भाषा धेरै मीठो र गम्भीर छ।",
]

for i, txt in enumerate(texts):
    print(f"Generating: {txt}")
    w = model.generate(txt, "ne", audio_prompt_path="ref.wav", exaggeration=0.5, temperature=0.8)
    ta.save(f"output_{i}.wav", w, model.sr)
    print(f"✅ output_{i}.wav — {w.shape[1]/model.sr:.1f}s")
    display(Audio(f"output_{i}.wav"))

In [ ]:
# 🎙️ Clone YOUR voice!
# Upload a 5-10 second audio file, then run this cell
from google.colab import files
uploaded = files.upload()  # Select your .wav or .mp3 file
my_voice_path = list(uploaded.keys())[0]

text = "यो मेरो आवाजमा बोलिएको नेपाली वाक्य हो।"
wav = model.generate(text, "ne", audio_prompt_path=my_voice_path, exaggeration=0.5, temperature=0.8)
ta.save("my_voice_output.wav", wav, model.sr)
print(f"✅ Cloned voice: {wav.shape[1]/model.sr:.1f}s")
display(Audio("my_voice_output.wav"))